In [1]:
# %% Libraries
import os
import scanpy as sc
#import scvi
import numpy as np
from pathlib import Path
import pandas as pd
#import novae
import matplotlib.pyplot as plt

import os
import plotnine as p9

# %% Setting Paths
MAIN_DIR_NAME = "proseg_data"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
CUR_OBJ_V = 'refined-manual-niches'
CUR_OBJ_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb-{CUR_OBJ_V}.h5ad'

# %% Load the object
comb = sc.read_h5ad(CUR_OBJ_PATH)



In [2]:
PLOTS_DIR = MAIN_DIR/ 'results/comb/plots/manual-niches'

In [3]:
comb

AnnData object with n_obs × n_vars = 1187341 × 2000
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'slide_ID', 'sample_name', 'n_genes', 'slide_name', 'condition', 'donor', 'run', 'modulator', 'age', 'mutation', 'sex', '_indices', '_scvi_batch', '_scvi_ind_x', '_scvi_labels', 'leiden_resolvi_0.1', 'leiden_resolvi_0.2', 'leiden_resolvi_0.3', 'leiden_resolvi_0.4', 'leiden_resolvi_0.5', 'leiden_resolvi_0.6', 'leiden_resolvi_0.7', 'leiden_resolvi_0.8', 'leiden_resolvi_0.9', 'leiden_resolvi_1.0', 'leiden_resolvi_1.1', 'leiden_resolvi_1.2', 'leiden_resolvi_1.3', 'leiden_resolvi_1.4', 'leiden_resolvi_1.5', 'ct_resolvi', 'ann_lvl_3', 'airway-epithelium_leiden_n10_0.4', 'ann_lvl_3_refined', 'ambiguous_leiden_n10_0.2', 'plasma_leiden_n10_0.3', 'T_leiden_n10_0.3', 'B_leiden_n10_0.2', 'endothelial_leiden_n10_0.2', 'ann_lvl_2', 'ann_lvl_1', 'niches_k_1', 'niches_k_2', 'niches_k_3', 'niches_k_4', 'niches_k_5', 'niches_k_

## get the proprotions

In [4]:
ct_labels = "ann_lvl_3_refined"
niche_col = "niche_name"

# 1) Subset obs columns
df = comb.obs[
    [ct_labels, niche_col]
].copy().dropna()

# 2) Count cells per niche × cell type
plot_data = (
    df.groupby(
        [niche_col, ct_labels],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# 3) Calculate cell-type proportions within each niche
plot_data["proportion"] = (
    plot_data["count"] /
    plot_data.groupby(niche_col)["count"].transform("sum")
)

plot_data['percentage'] = plot_data['proportion']*100
airway_comp = (
    plot_data[
        plot_data["niche_name"] == "airway_epithelium"
    ]
    .sort_values("percentage", ascending=False)
)

airway_comp

/tmp/ipykernel_4028345/3714486671.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,niche_name,ann_lvl_3_refined,count,proportion,percentage
72,airway_epithelium,MCC,72413,0.307317,30.731656
63,airway_epithelium,BC,50243,0.213228,21.322837
70,airway_epithelium,Gob,39248,0.166566,16.656623
66,airway_epithelium,Club,34238,0.145304,14.530408
75,airway_epithelium,Neut,8447,0.035849,3.584858
73,airway_epithelium,Mac,7037,0.029865,2.986462
61,airway_epithelium,Amb,4587,0.019467,1.946696
74,airway_epithelium,Mast,3292,0.013971,1.397106
59,airway_epithelium,AT2,2465,0.010461,1.046132
69,airway_epithelium,Fib,2452,0.010406,1.040615


## get the proportions per condition

In [5]:
ct_labels = "ann_lvl_3_refined"
niche_col = "niche_name"
condition_col = "condition"

# 1) Subset obs columns
df = comb.obs[
    [ct_labels, niche_col, condition_col]
].copy().dropna()

# 2) Count cells per niche × condition × cell type
plot_data = (
    df.groupby(
        [niche_col, condition_col, ct_labels],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# 3) Calculate cell-type proportions within each niche × condition
plot_data["proportion"] = (
    plot_data["count"] /
    plot_data.groupby(
        [niche_col, condition_col]
    )["count"].transform("sum")
)

plot_data["percentage"] = (
    plot_data["proportion"] * 100
)

# 4) Airway epithelial niche only
airway_comp = (
    plot_data[
        plot_data[niche_col] == "airway_epithelium"
    ]
    .sort_values(
        [condition_col, "percentage"],
        ascending=[True, False]
    )
)

airway_comp

/tmp/ipykernel_4028345/1980383962.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,niche_name,condition,ann_lvl_3_refined,count,proportion,percentage
184,airway_epithelium,CF,MCC,61841,0.321914,32.191417
175,airway_epithelium,CF,BC,45442,0.236549,23.654895
182,airway_epithelium,CF,Gob,33574,0.174770,17.476992
178,airway_epithelium,CF,Club,19024,0.099030,9.902969
187,airway_epithelium,CF,Neut,7340,0.038208,3.820847
...,...,...,...,...,...,...
248,airway_epithelium,CTRL,PC_L_IgA,7,0.000278,0.027818
245,airway_epithelium,CTRL,PC_K_IgA,6,0.000238,0.023844
246,airway_epithelium,CTRL,PC_K_IgG,3,0.000119,0.011922
247,airway_epithelium,CTRL,PC_K_IgM,1,0.000040,0.003974


In [8]:
airway_comp[airway_comp['condition']=='CF']

,niche_name,condition,ann_lvl_3_refined,count,proportion,percentage
184,airway_epithelium,CF,MCC,61841,0.321914,32.191417
175,airway_epithelium,CF,BC,45442,0.236549,23.654895
182,airway_epithelium,CF,Gob,33574,0.174770,17.476992
178,airway_epithelium,CF,Club,19024,0.099030,9.902969
187,airway_epithelium,CF,Neut,7340,0.038208,3.820847
185,airway_epithelium,CF,Mac,6021,0.031342,3.134240
173,airway_epithelium,CF,Amb,3435,0.017881,1.788094
186,airway_epithelium,CF,Mast,2677,0.013935,1.393516
171,airway_epithelium,CF,AT2,1887,0.009823,0.982280
183,airway_epithelium,CF,Hillock,1644,0.008558,0.855786


In [9]:
airway_comp[airway_comp['condition']=='CTRL']

,niche_name,condition,ann_lvl_3_refined,count,proportion,percentage
235,airway_epithelium,CTRL,Club,12754,0.506835,50.683516
241,airway_epithelium,CTRL,MCC,5018,0.199412,19.941186
232,airway_epithelium,CTRL,BC,2315,0.091997,9.199650
230,airway_epithelium,CTRL,Amb,931,0.036997,3.699730
238,airway_epithelium,CTRL,Fib,780,0.030997,3.099666
239,airway_epithelium,CTRL,Gob,661,0.026268,2.626768
228,airway_epithelium,CTRL,AT2,467,0.018558,1.855826
244,airway_epithelium,CTRL,Neut,428,0.017008,1.700842
242,airway_epithelium,CTRL,Mac,385,0.015300,1.529963
250,airway_epithelium,CTRL,SMC,308,0.012240,1.223971
